In [ ]:
!git clone https://github.com/deepanrajm/GL_MML.git

In [17]:

import os
import librosa
import tensorflow as tf
import numpy as np
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Dropout
from keras.utils import to_categorical
from keras.models import load_model
import matplotlib.pyplot as plt
import seaborn as sns
import librosa.display

In [4]:
label = ["car_horn","dog_bark","engine_idling","siren"]

In [5]:

car_horn_onehot 		= [1,0,0,0]
dog_bark_onehot 		= [0,1,0,0]
engine_idling_onehot 	= [0,0,1,0]
siren_onehot 			= [0,0,0,1]

In [19]:

def extract_feature(file_name):
	print("Extracting "+file_name+" ...")
	X, sample_rate = librosa.load(file_name)
	stft = np.abs(librosa.stft(X))
	mfccs = np.mean(librosa.feature.mfcc(y=X, sr=sample_rate, n_mfcc=40).T,axis=0)
	chroma = np.mean(librosa.feature.chroma_stft(S=stft, sr=sample_rate).T,axis=0)
	mel = np.mean(librosa.feature.melspectrogram(y=X, sr=sample_rate).T, axis=0)
	contrast = np.mean(librosa.feature.spectral_contrast(S=stft, sr=sample_rate).T,axis=0)
	tonnetz = np.mean(librosa.feature.tonnetz(y=librosa.effects.harmonic(X),sr=sample_rate).T,axis=0)
	return mfccs,chroma,mel,contrast,tonnetz,X, sample_rate


In [ ]:

def visualize_audio_embedding(file_path):
    mfccs, chroma, mel, contrast, tonnetz, X, sample_rate = extract_feature(file_path)


    full_embedding = np.hstack((mfccs, chroma, mel, contrast, tonnetz))

    plt.figure(figsize=(16, 10))


    plt.subplot(3, 2, 1)
    librosa.display.waveshow(X, sr=sample_rate)
    plt.title("1. Raw Time-Domain Waveform")

    plt.subplot(3, 2, 2)
    sns.heatmap(full_embedding.reshape(1, -1), cmap='magma', cbar=False, yticklabels=False)
    plt.title("2. The Full 193-Feature Embedding (AI Input)")


    plt.subplot(3, 2, 3)
    plt.plot(mfccs)
    plt.title("3. MFCC Coefficients (Timbre Shape)")


    plt.subplot(3, 2, 4)
    plt.bar(range(12), chroma)
    plt.xticks(range(12), ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B'])
    plt.title("4. Chroma Representation (Pitch Content)")


    plt.subplot(3, 2, 5)
    S_dB = librosa.power_to_db(librosa.feature.melspectrogram(y=X, sr=sample_rate), ref=np.max)
    librosa.display.specshow(S_dB, x_axis='time', y_axis='mel')
    plt.title("5. Mel-Spectrogram (Energy over Frequency)")

    plt.tight_layout()
    plt.show()


test_file = "/content/GL_MML/Unit_2/Sound_Representation/siren/98536.wav"
visualize_audio_embedding(test_file)

In [ ]:
def decodeFolder(category):
    print("Starting decoding folder "+category+" ...")
    listOfFiles = os.listdir(category)
    arrays_sound_features = np.empty((0,193))
    for file in listOfFiles:
        filename = os.path.join(category,file)


        mfccs, chroma, mel, contrast, tonnetz, _, _ = extract_feature(filename)

        single_feature_vector = np.hstack((mfccs, chroma, mel, contrast, tonnetz))

        arrays_sound_features = np.vstack((arrays_sound_features, single_feature_vector))
    return arrays_sound_features


car_horn_sounds = decodeFolder("/content/GL_MML/Unit_2/Sound_Representation/car_horn")
car_horn_labels = [car_horn_onehot for _ in car_horn_sounds]

dog_bark_sounds = decodeFolder("/content/GL_MML/Unit_2/Sound_Representation/dog_bark")
dog_bark_labels = [dog_bark_onehot for _ in dog_bark_sounds]

engine_idling_sounds = decodeFolder("/content/GL_MML/Unit_2/Sound_Representation/engine_idling")
engine_idling_labels = [engine_idling_onehot for _ in engine_idling_sounds]

siren_sounds = decodeFolder("/content/GL_MML/Unit_2/Sound_Representation/siren")
siren_labels = [siren_onehot for _ in siren_sounds]

In [ ]:
train_sounds = np.concatenate((car_horn_sounds, dog_bark_sounds,engine_idling_sounds,siren_sounds))
train_labels = np.concatenate((car_horn_labels, dog_bark_labels,engine_idling_labels,siren_labels))
print (train_sounds.shape)
X_train = train_sounds.reshape(train_sounds.shape[0], train_sounds.shape[1]).astype('float32')


test_sound = decodeFolder("/content/GL_MML/Unit_2/Sound_Representation/test")

print (test_sound.shape)
X_test = test_sound.reshape(test_sound.shape[0], test_sound.shape[1]).astype('float32')


In [ ]:
model = Sequential()
model.add(Dense(193, input_dim=193, activation='relu'))

model.add(Dense(4, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.fit(X_train, train_labels, epochs=150, batch_size=10)

model.save('my_model.h5')

In [ ]:
model = load_model('my_model.h5')

In [ ]:
predictions = model.predict(X_test)
pred = np.argmax(predictions, axis=1)

In [33]:
listOfFiles = os.listdir("/content/GL_MML/Unit_2/Sound_Representation/test")

In [ ]:
for i in range (0, len(listOfFiles)):
	print ("Listening to",listOfFiles[i] )

	print ("I think it is", label[pred[i]],"sound")

In [ ]:
import tensorflow_hub as hub


yamnet_model_handle = 'https://tfhub.dev/google/yamnet/1'
yamnet_model = hub.load(yamnet_model_handle)

def visualize_yamnet_feature_maps(file_path):

    waveform, sr = librosa.load(file_path, sr=16000)

    scores, embeddings, spectrogram_map = yamnet_model(waveform)

    plt.figure(figsize=(15, 10))


    plt.subplot(3, 1, 1)
    plt.imshow(spectrogram_map.numpy().T, aspect='auto', interpolation='nearest', origin='lower')
    plt.title("1. Input Audio Representation (Log-Mel Spectrogram)")
    plt.ylabel("Frequency Bins")


    plt.subplot(3, 1, 2)
    plt.imshow(embeddings.numpy().T, aspect='auto', interpolation='nearest', origin='lower', cmap='magma')
    plt.title("2. Deep Acoustic Embeddings (Learned Representation)")
    plt.ylabel("Latent Features")

    plt.subplot(3, 1, 3)
    plt.imshow(scores.numpy().T, aspect='auto', interpolation='nearest', origin='lower', cmap='viridis')
    plt.title("3. Output Class Activation (Which sound is it?)")
    plt.xlabel("Time Segments")
    plt.ylabel("Class Index")

    plt.tight_layout()
    plt.show()

visualize_yamnet_feature_maps("/content/GL_MML/Unit_2/Sound_Representation/siren/98536.wav")